<a href="https://colab.research.google.com/github/Zaheem1/PDF-QA-System-using-RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain
!pip install transformers

In [2]:
!pip install fiass-cpu
!pip install PyMuPDF

ERROR: Could not find a version that satisfies the requirement fiass-cpu (from versions: none)
ERROR: No matching distribution found for fiass-cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 68.3 MB/s eta 0:00:00


In [3]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.2 MB/s eta 0:00:00


In [9]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import fitz

In [10]:
!pip install pymupdf

In [11]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import fitz

In [17]:
from google.colab import files
uploaded = files.upload()


Saving testfile.pdf to testfile.pdf


In [18]:
import fitz

def load_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""

    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        text += page.get_text()

    return text


pdf_text = load_pdf("testfile.pdf")

print(pdf_text[:1000])  # preview first part

 
 
 
 
 
 
 
 
 
 
AGREEMENT (SAMPLE) 
 
 
 
 
This agreement (“Agreement”) is made on this the ________ day of ---------------- at Islamabad. 
 
 
BY AND BETWEEN 
 
Securities and Exchange Commission of Pakistan, a statutory body established in pursuance of the Securities and 
Exchange Commission of Pakistan Act, 1997 with its head office located at NIC Building, 63 Jinnah Avenue, 
Islamabad (the "Commission", which expression shall, where the context so admits, include its successors in interest and 
permitted assigns) of the One Part 
AND 
 
(Name of Firm), a firm in the field of design & build solutions, registered as a sole Proprietor/ partnership 
under the __________Act, 1932, having its office at (Office Address) ( the “Contractor” which expression shall, 
where the context so admits, include its successors in interest and permitted assigns) of the other part; 
 
Commission and the Contractor shall hereinafter be referred to as the “Parties” collectively and the “Party” indivi

In [19]:
def chunk_text(text, size=500):
    chunks = []
    for i in range(0, len(text), size):
        chunks.append(text[i:i+size])
    return chunks

chunks = chunk_text(pdf_text)

In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

In [22]:
from transformers import pipeline

qa_pipeline = pipeline("question-answering")

def ask_question(question):
    # Convert question to embedding
    q_embedding = model.encode([question])

    # Search similar chunks
    D, I = index.search(np.array(q_embedding), k=3)

    context = " ".join([chunks[i] for i in I[0]])

    # Get answer
    result = qa_pipeline(question=question, context=context)
    return result["answer"]

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [23]:
question = "What is Amicable settlement"
answer = ask_question(question)

print("Answer:", answer)

Answer: ment


In [24]:
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [25]:
def chunk_text(text, size=500, overlap=100):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i+size])
    return chunks

In [27]:
pdf_text = pdf_text.replace("\n", " ")

In [30]:
def ask_question(question):
    q_embedding = model.encode([question])

    D, I = index.search(np.array(q_embedding), k=5)

    context = " ".join([chunks[i] for i in I[0]])

    result = qa_pipeline(question=question, context=context)

    return result["answer"]

In [31]:
question = "What is Amicable settlement"
answer = ask_question(question)

print("Answer:", answer)

Answer: 
ARTICLE 12 – ENTIRETY AND COUNTERPARTS
